In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib  # For saving model


In [2]:
# Load the dataset
df = pd.read_csv("survey.csv")
print(df.shape)
df.head()


(1259, 27)


,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:31,37,Female,United States,IL,NaN,No,Yes,Often,6-25,...,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,2014-08-27 11:29:37,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,...,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,2014-08-27 11:29:44,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,...,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,2014-08-27 11:29:46,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,...,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,2014-08-27 11:30:22,31,Male,United States,TX,NaN,No,No,Never,100-500,...,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN


In [3]:
# Drop columns not useful for prediction (e.g., timestamp, comments)
df.drop(columns=["Timestamp", "comments", "state"], inplace=True, errors='ignore')


In [4]:
# Check missing values
print(df.isnull().sum())

# Fill missing values for categorical features with most frequent value
cat_imputer = SimpleImputer(strategy='most_frequent')
cat_cols = df.select_dtypes(include='object').columns
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# Fill missing values for numerical columns with mean
num_imputer = SimpleImputer(strategy='mean')
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = num_imputer.fit_transform(df[num_cols])


Age                            0
Gender                         0
Country                        0
self_employed                 18
family_history                 0
treatment                      0
work_interfere               264
no_employees                   0
remote_work                    0
tech_company                   0
benefits                       0
care_options                   0
wellness_program               0
seek_help                      0
anonymity                      0
leave                          0
mental_health_consequence      0
phys_health_consequence        0
coworkers                      0
supervisor                     0
mental_health_interview        0
phys_health_interview          0
mental_vs_physical             0
obs_consequence                0
dtype: int64


In [5]:
# Encode binary categorical columns using LabelEncoder
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])


In [6]:
# Define your features (X) and target (y)
# Example target: 'treatment' — whether a person sought mental health treatment
X = df.drop(columns=["treatment"])
y = df["treatment"]  # Already encoded as 0/1 in the step above


In [7]:
# Scale numerical data for better accuracy
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (1007, 23)
Test size: (252, 23)


In [9]:
model = LogisticRegression(max_iter=1000)  # You can tune max_iter, penalty, C, etc.
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

In [10]:
y_pred = model.predict(X_test)


In [11]:
# Accuracy score
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy: {:.2f}%".format(accuracy * 100))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)

# Classification report
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Test Accuracy: 70.24%

Confusion Matrix:
 [[93 31]
 [44 84]]

Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.75      0.71       124
           1       0.73      0.66      0.69       128

    accuracy                           0.70       252
   macro avg       0.70      0.70      0.70       252
weighted avg       0.71      0.70      0.70       252



In [12]:
joblib.dump(model, 'mental_health_model.pkl')



['mental_health_model.pkl']